## 1. Setup Environment
Mount Google Drive to access `dataset.zip` and `best_model.pt`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Copy and Unzip Dataset
Assuming `dataset.zip` and `best_model.pt` are in the root of your Google Drive. Adjust the path if they are in a folder (e.g., `/content/drive/MyDrive/satellite_project/`).


In [ ]:
!cp "/content/drive/MyDrive/dataset.zip" /content/
!cp "/content/drive/MyDrive/best_model.pt" /content/
!unzip -q /content/dataset.zip -d /content/


## 3. Define the Model (MobileNetV2 3D CDNet)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class Fusion3D(nn.Module):
    """
    3D Convolution fusion module inspired by AFCF3D-Net.
    Fuses temporal features (t=2) using a 3D convolution to capture spatial-temporal relations.
    """
    def __init__(self, channels):
        super(Fusion3D, self).__init__()
        # Depthwise 3D Conv (groups=channels) to drastically save parameters
        self.conv3d = nn.Conv3d(channels, channels, kernel_size=(2, 3, 3), padding=(0, 1, 1), groups=channels)
        # Pointwise Conv to mix channels after spatial/temporal fusion
        self.pointwise = nn.Conv2d(channels, channels, kernel_size=1)
        
        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, f1, f2):
        # f1, f2 shape: [B, C, H, W]
        # Stack along temporal dimension: [B, C, 2, H, W]
        x = torch.stack((f1, f2), dim=2)
        
        # 3D Conv: [B, C, 1, H, W]
        x = self.conv3d(x)
        
        # Squeeze temporal dimension: [B, C, H, W]
        x = x.squeeze(2)
        
        # Pointwise mix
        x = self.pointwise(x)
        
        x = self.bn(x)
        x = self.relu(x)
        return x

class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Pad if sizes don't match
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MobileNetV2_3D_CDNet(nn.Module):
    """
    Advanced Change Detection architecture:
    - MobileNetV2 backbone (shared weights)
    - 3D Convolution Fusion (AFCF3D-Net inspired)
    - U-Net style decoder
    """
    def __init__(self, n_classes=1):
        super(MobileNetV2_3D_CDNet, self).__init__()
        
        # Pretrained MobileNetV2 backbone
        import ssl
        _orig_ctx = ssl._create_default_https_context
        try:
            ssl._create_default_https_context = ssl._create_unverified_context  # type: ignore[assignment]
            mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT).features
        except Exception:
            mobilenet = models.mobilenet_v2(pretrained=True).features
        finally:
            ssl._create_default_https_context = _orig_ctx
            
        # MobileNetV2 feature extractor
        self.inc = nn.Sequential(*mobilenet[:2])     # 1/2, 16 channels
        self.down1 = nn.Sequential(*mobilenet[2:4])  # 1/4, 24 channels
        self.down2 = nn.Sequential(*mobilenet[4:7])  # 1/8, 32 channels
        self.down3 = nn.Sequential(*mobilenet[7:14]) # 1/16, 96 channels
        self.down4 = nn.Sequential(*mobilenet[14:])  # 1/32, 1280 channels
        
        # 3D Fusion modules at each scale
        self.fuse1 = Fusion3D(16)
        self.fuse2 = Fusion3D(24)
        self.fuse3 = Fusion3D(32)
        self.fuse4 = Fusion3D(96)
        self.fuse5 = Fusion3D(1280)
        
        # Decoder
        self.up1 = Up(1280 + 96, 256)
        self.up2 = Up(256 + 32, 128)
        self.up3 = Up(128 + 24, 64)
        self.up4 = Up(64 + 16, 32)
        
        # Final upsampling to original size (since inc is 1/2)
        self.up5 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            DoubleConv(32, 16)
        )
        
        self.outc = nn.Conv2d(16, n_classes, kernel_size=1)

    def forward_encoder(self, x):
        x1 = self.inc(x)      
        x2 = self.down1(x1)   
        x3 = self.down2(x2)   
        x4 = self.down3(x3)   
        x5 = self.down4(x4)   
        return x1, x2, x3, x4, x5

    def forward(self, x_a, x_b):
        a1, a2, a3, a4, a5 = self.forward_encoder(x_a)
        b1, b2, b3, b4, b5 = self.forward_encoder(x_b)
        
        # 3D Fusion
        f1 = self.fuse1(a1, b1)
        f2 = self.fuse2(a2, b2)
        f3 = self.fuse3(a3, b3)
        f4 = self.fuse4(a4, b4)
        f5 = self.fuse5(a5, b5)
        
        # Decoder
        x = self.up1(f5, f4)
        x = self.up2(x, f3)
        x = self.up3(x, f2)
        x = self.up4(x, f1)
        
        x = self.up5(x)
        logits = self.outc(x)
        return logits



## 4. Run the Degradation Experiment


In [ ]:
import os
import io
import torch
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
from tqdm import tqdm

def preprocess_numpy(img_rgb):
    img_array = img_rgb.astype(np.float32) / 255.0
    img_array = np.transpose(img_array, (2, 0, 1)) # HWC to CHW
    
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)
    img_array = (img_array - mean) / std
    return img_array

def degrade_image(img_arr, factor):
    if factor == 1:
        return img_arr
    
    img = Image.fromarray(img_arr)
    sigma = factor / 2.0
    blurred = img.filter(ImageFilter.GaussianBlur(radius=sigma))
    
    w, h = img.size
    new_w, new_h = w // factor, h // factor
    downsampled = blurred.resize((new_w, new_h), resample=Image.LANCZOS)
    upsampled = downsampled.resize((w, h), resample=Image.LANCZOS)
    return np.array(upsampled)

def run_degradation_experiment():
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"[device] {device}")

    model = MobileNetV2_3D_CDNet(n_classes=1).to(device)
    state = torch.load("best_model.pt", map_location=device)
    model.load_state_dict(state)
    model.eval()

    factors = [1, 2, 4, 8, 16, 20]
    resolutions = {1: "0.5m", 2: "1.0m", 4: "2.0m", 8: "4.0m", 16: "8.0m", 20: "10.0m"}
    
    test_dir_A = "dataset/test/A"
    test_dir_B = "dataset/test/B"
    test_dir_label = "dataset/test/label"
    
    files = sorted(os.listdir(test_dir_A))
    results = []
    
    with torch.no_grad():
        for filename in tqdm(files, desc="Processing images"):
            if not filename.endswith((".png", ".jpg")):
                continue
                
            path_a = os.path.join(test_dir_A, filename)
            path_b = os.path.join(test_dir_B, filename)
            path_label = os.path.join(test_dir_label, filename)
            
            img_a = np.array(Image.open(path_a).convert("RGB"))
            img_b = np.array(Image.open(path_b).convert("RGB"))
            label = np.array(Image.open(path_label).convert("L"))
            gt_mask = (label > 0).astype(np.uint8)
            
            if img_a.shape[:2] != (256, 256):
                img_a = np.array(Image.fromarray(img_a).resize((256, 256), resample=Image.LANCZOS))
                img_b = np.array(Image.fromarray(img_b).resize((256, 256), resample=Image.LANCZOS))
                gt_mask = np.array(Image.fromarray(gt_mask).resize((256, 256), resample=Image.NEAREST))
                
            batch_a = []
            batch_b = []
            
            for factor in factors:
                deg_a = degrade_image(img_a, factor)
                deg_b = degrade_image(img_b, factor)
                batch_a.append(preprocess_numpy(deg_a))
                batch_b.append(preprocess_numpy(deg_b))
                
            # Stack along batch dimension (since each is [3, 256, 256])
            t_a = torch.tensor(np.stack(batch_a)).to(device)
            t_b = torch.tensor(np.stack(batch_b)).to(device)
            
            logits = model(t_a, t_b) # [6, 1, 256, 256]
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(np.uint8)[:, 0] # [6, 256, 256]
            
            for i, factor in enumerate(factors):
                pred_mask = preds[i]
                
                tp = np.sum((pred_mask == 1) & (gt_mask == 1))
                fp = np.sum((pred_mask == 1) & (gt_mask == 0))
                fn = np.sum((pred_mask == 0) & (gt_mask == 1))
                
                precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
                f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
                iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
                
                results.append({
                    "image": filename,
                    "factor": factor,
                    "real_world_resolution": resolutions[factor],
                    "precision": precision,
                    "recall": recall,
                    "f1": f1,
                    "iou": iou,
                    "tp": tp,
                    "fp": fp,
                    "fn": fn
                })

    df = pd.DataFrame(results)
    df.to_csv("degradation_results.csv", index=False)
    
    grouped = df.groupby('factor')[['tp', 'fp', 'fn']].sum()
    
    df_positive = df[df['tp'] + df['fn'] > 0]
    macro_stats = df_positive.groupby('factor').agg(
        iou_mean=('iou', 'mean'),
        iou_std=('iou', 'std'),
        f1_mean=('f1', 'mean'),
        f1_std=('f1', 'std'),
        precision_mean=('precision', 'mean'),
        precision_std=('precision', 'std'),
        recall_mean=('recall', 'mean'),
        recall_std=('recall', 'std')
    )
    
    print("\n=== DEGRADATION SUMMARY ===")
    for factor in factors:
        res = resolutions[factor]
        tps = grouped.loc[factor, 'tp']
        fps = grouped.loc[factor, 'fp']
        fns = grouped.loc[factor, 'fn']
        
        micro_prec = tps / (tps + fps) if (tps + fps) > 0 else 0.0
        micro_rec = tps / (tps + fns) if (tps + fns) > 0 else 0.0
        micro_f1 = 2 * (micro_prec * micro_rec) / (micro_prec + micro_rec) if (micro_prec + micro_rec) > 0 else 0.0
        micro_iou = tps / (tps + fps + fns) if (tps + fps + fns) > 0 else 0.0
        
        print(f"\nFactor {factor}x ({res}):")
        print(f"  Micro-Avg -> F1: {micro_f1:.4f} | IoU: {micro_iou:.4f} | Prec: {micro_prec:.4f} | Rec: {micro_rec:.4f}")
        if factor in macro_stats.index:
            row = macro_stats.loc[factor]
            print(f"  Macro-Avg -> F1: {row['f1_mean']:.4f} ± {row['f1_std']:.4f} | IoU: {row['iou_mean']:.4f} ± {row['iou_std']:.4f}")

if __name__ == "__main__":
    run_degradation_experiment()

